# Einstieg in JSON

[JSON](https://www.json.org/json-en.html) ist eines der häufigsten Formate, denen du bei der Arbeit mit APIs, Konfigurationsdateien und Datenpipelines begegnest. In diesem Notebook bauen wir zuerst ein einfaches mentales Modell auf und üben dann den Umgang mit den Tools, die das offizielle [`json`-Modul](https://docs.python.org/3/library/json.html) von Python bietet.

Am Ende dieses Notebooks kannst du:
- erklären, was JSON ist und warum APIs es verwenden,
- Python-Objekte in JSON-Strings umwandeln und umgekehrt,
- JSON in einer Datei speichern und wieder laden,
- eingelesene JSON-Daten untersuchen und aktualisieren,
- und JSON mit Pydantic-Modellen validieren.


> **Was du schon wissen solltest:** Du solltest sicher mit einfachen Python-Variablen, Listen und Dictionarys umgehen können.


## Visueller Überblick

```mermaid
graph TD
    JSON[JSON-Text]
    PY[Python-Objekte]
    FILE[JSON-Datei]
    API[API-Response]
    MODEL[Pydantic-Modell]

    JSON -->|json.loads| PY
    FILE -->|json.load| PY
    PY -->|json.dump| FILE
    PY -->|json.dumps| JSON
    API -->|response.json| PY
    PY -->|validieren| MODEL
```


In [ ]:
import json
from pathlib import Path

import requests
from pydantic import BaseModel, ValidationError

## Mentales Modell für JSON

JSON steht für **JavaScript Object Notation**. Es ist ein Textformat zum Speichern und Austauschen strukturierter Daten, deshalb taucht es so oft in APIs auf.

Ein gutes erstes mentales Modell ist: **JSON ist Text**, während **Python-Dictionarys und -Listen Python-Objekte im Arbeitsspeicher sind**.

Hier ist ein kleines JSON-Beispiel:

```json
{
  "name": "Ada Lovelace",
  "age": 36,
  "skills": ["math", "writing"],
  "active": true,
  "manager": null
}
```

Wenn Python JSON liest, bildet es die Werte so ab:

| JSON | Python |
| --- | --- |
| Objekt | `dict` |
| Array | `list` |
| String | `str` |
| Zahl | `int` oder `float` |
| true / false | `True` / `False` |
| null | `None` |

Beachte, dass das JSON-Beispiel kleingeschriebene `true` und `null` verwendet. Im Python-Code sind die entsprechenden Werte `True` und `None`.


## Erst mit JSON im Arbeitsspeicher arbeiten

Warum hier anfangen? Weil du mit `dumps()` und `loads()` die Umwandlung selbst verstehen kannst, bevor Dateien ins Spiel kommen.

- `json.dumps(...)` wandelt ein Python-Objekt in einen JSON-**String** um
- `json.loads(...)` wandelt einen JSON-**String** in ein Python-Objekt um


In [ ]:
profile = {
    "name": "Ada Lovelace",
    "age": 36,
    "skills": ["math", "writing"],
    "active": True,
    "manager": None,
}

profile_json = json.dumps(profile, indent=2)
profile_roundtrip = json.loads(profile_json)

print("Python object type:", type(profile).__name__)
print("JSON string type:", type(profile_json).__name__)
print("Round-trip type:", type(profile_roundtrip).__name__)
print()
print(profile_json)

**Erwartetes Ergebnis:** Du solltest einen schön formatierten JSON-String sehen. Der ursprüngliche Wert ist ein Python-`dict`, aber `profile_json` ist ein einfacher String, der zufällig JSON-Text enthält.


In [ ]:
blackjack_hand = (8, "Q")
encoded_hand = json.dumps(blackjack_hand)
decoded_hand = json.loads(encoded_hand)

print("Original value:", blackjack_hand, type(blackjack_hand).__name__)
print("After dumps():", encoded_hand, type(encoded_hand).__name__)
print("After loads():", decoded_hand, type(decoded_hand).__name__)
print("Same value without type conversion?", blackjack_hand == decoded_hand)
print(
    "Same value after converting back to tuple?", blackjack_hand == tuple(decoded_hand)
)

> **Checkpoint:**
> Bevor du weitermachst, stell sicher, dass dir dieser Unterschied klar ist:
> - `dumps()` und `loads()` arbeiten mit **Strings im Arbeitsspeicher**
> - nach einer Umwandlung in JSON und zurück können manche Python-Typen leicht verändert zurückkommen, zum Beispiel wird ein Tupel zu einer Liste


## JSON in einer Datei speichern und wieder laden

Da die Umwandlung mit Strings jetzt klar ist, können wir das Lesen und Schreiben von Dateien dazunehmen.

- `json.dump(...)` schreibt JSON in eine Datei
- `json.load(...)` liest JSON aus einer Datei


In [ ]:
Path("data").mkdir(exist_ok=True)

persons_seed = [
    {
        "name": "John Doe",
        "age": 30,
        "email": "john.doe@example.com",
        "address": {
            "street": "123 Main St",
            "city": "Anytown",
            "state": "CA",
            "zip": "12345",
        },
        "phoneNumbers": [
            {"type": "home", "number": "555-555-1234"},
            {"type": "work", "number": "555-555-5678"},
        ],
        "isEmployee": True,
        "salary": None,
    }
]

with open("data/example.json", "w") as f:
    json.dump(persons_seed, f, indent=2)

with open("data/example.json") as f:
    persons = json.load(f)

print("File written to:", Path("data/example.json").resolve())
print("Loaded object type:", type(persons).__name__)
print("First record:")
print(json.dumps(persons[0], indent=2))

**Erwartetes Ergebnis:** Das Notebook sollte `data/example.json` erstellen und die Datei dann wieder als Python-Liste von Dictionarys laden.


## Eingelesenes JSON lesen und ändern

Sobald JSON in Python geladen ist, bearbeitest du es mit normalen Python-Operationen für Listen und Dictionarys.


In [ ]:
print("Name:", persons[0].get("name"))
print("First phone number:", persons[0]["phoneNumbers"][0]["number"])

persons[0].update({"name": "Jane Doe"})

new_person = {
    "name": "Sam Rivera",
    "age": 25,
    "email": "sam.rivera@example.com",
    "address": {
        "street": "456 Main St",
        "city": "Anytown",
        "state": "CA",
        "zip": "12345",
    },
    "phoneNumbers": [{"type": "mobile", "number": "555-555-4321"}],
    "isEmployee": False,
    "salary": None,
}

persons.append(new_person)

print()
print("Updated first name:", persons[0]["name"])
print("Number of people:", len(persons))
print("Second person name:", persons[1]["name"])

**Erwartetes Ergebnis:** Du solltest Werte sehen, die aus verschachtelten JSON-Daten ausgelesen wurden, einen aktualisierten Namen und eine zweite Person, die zur Liste hinzugefügt wurde.

> **Checkpoint:**
> Wenn `persons` aus `json.load(...)` stammt, warum kannst du darauf die Indexierung von Listen und Methoden von Dictionarys anwenden?  
> **Antwort:** Weil JSON-Daten nach dem Parsing zu normalen Python-Objekten werden.


## Ein Beispiel mit einer echten API

Die meisten APIs senden JSON in ihren Responses. Mit der Bibliothek [Requests](https://requests.readthedocs.io/en/latest/) kannst du diese Daten leicht abrufen.

In der Praxis ist `response.json()` der übliche, bequeme Weg, eine JSON-Response einzulesen. `json.loads(response.text)` ist der explizitere Low-Level-Weg, wenn du selbst vom Rohtext ausgehen möchtest.


In [ ]:
response = requests.get("https://jsonplaceholder.typicode.com/todos", timeout=10)
response.raise_for_status()

todos_from_response = response.json()
todos_from_text = json.loads(response.text)

print("Status code:", response.status_code)
print("Objects match:", todos_from_response == todos_from_text)
print("Total todos:", len(todos_from_response))
print("First todo:")
print(json.dumps(todos_from_response[0], indent=2))

**Erwartetes Ergebnis:** Beide Parsing-Ansätze sollten dieselben Python-Daten erzeugen. Bevorzuge im Alltagscode `response.json()`, außer du brauchst gezielt zuerst den Rohtext der Response.


## JSON mit Pydantic validieren

Das Parsing zeigt dir, wie du JSON in Python-Objekte umwandelst. [Pydantic-Modelle](https://docs.pydantic.dev/latest/concepts/models/) beantworten eine andere Frage: **Passen diese Daten zu der Form, die ich erwarte?**

Das ist bei APIs wichtig, weil in eingehenden oder externen Daten Felder fehlen können, falsche Typen vorkommen oder Werte stehen können, mit denen du nicht gerechnet hast.


In [ ]:
class Address(BaseModel):
    street: str
    city: str
    state: str
    zip: str


class User(BaseModel):
    name: str
    age: int
    email: str
    address: Address
    phone: str


valid_user_json = """{
    "name": "John Doe",
    "age": 30,
    "email": "john.doe@example.com",
    "address": {
        "street": "123 Main St",
        "city": "Anytown",
        "state": "CA",
        "zip": "12345"
    },
    "phone": "555-555-1234"
}"""

invalid_user_json = """{
    "name": "Broken Example",
    "age": "thirty",
    "email": "broken@example.com",
    "address": {
        "street": "123 Main St",
        "city": "Anytown",
        "state": "CA",
        "zip": 12345
    }
}"""

In [ ]:
valid_user = User.model_validate_json(valid_user_json)
print("Validated user:")
print(valid_user)
print()

valid_user_dict = json.loads(valid_user_json)
validated_from_dict = User.model_validate(valid_user_dict)
print("Validated from Python dict:")
print(validated_from_dict)
print()

print("Invalid example errors:")
try:
    User.model_validate_json(invalid_user_json)
except ValidationError as e:
    print(e)

**Erwartetes Ergebnis:** Das gültige Beispiel sollte sich in ein `User`-Objekt umwandeln lassen, während das ungültige Beispiel gut lesbare Validierungsfehler anzeigen sollte.

> **Checkpoint:**
> Versuch jetzt, den Unterschied laut auszusprechen:
> - Parsing wandelt Daten zwischen Formaten um
> - Validierung prüft, ob die Daten nach dem Parsing zum erwarteten Schema passen


## Rückblick

Diese wichtigen Unterschiede solltest du auseinanderhalten:

- **JSON vs Python-Objekte:** JSON ist Text; Python-Dictionarys und -Listen liegen im Arbeitsspeicher.
- **`dump/load` vs `dumps/loads`:** Die Versionen ohne `s` arbeiten mit Dateien, die Versionen mit `s` mit Strings.
- **`response.json()` vs `json.loads(response.text)`:** Beide können JSON aus einer API einlesen, aber `response.json()` ist meist die einfachere Wahl.
- **Parsing vs Validierung:** Das Parsing wandelt Daten um, die Validierung prüft, ob die Daten zur erwarteten Struktur passen.


## Übung

Probiere diesen Workflow von Anfang bis Ende aus:
1. Rufe Todos von JSONPlaceholder ab.
2. Untersuche einen Eintrag.
3. Definiere ein Pydantic-Modell für ein Todo.
4. Validiere eine kleine Teilmenge der Response.
5. Speichere die validierten Todos in einer JSON-Datei.


**Erwartetes Ergebnis:** Du solltest die ersten fünf Todos erfolgreich validieren und `data/validated_todos.json` erstellen.


In [ ]:
# Dein Code hier!


<details>
<summary>Musterlösung anzeigen</summary>

```python
class Todo(BaseModel):
    userId: int
    id: int
    title: str
    completed: bool


sample_todo = todos_from_response[0]
validated_todos = [Todo.model_validate(item) for item in todos_from_response[:5]]
validated_todo_dicts = [todo.model_dump() for todo in validated_todos]

with open("data/validated_todos.json", "w") as f:
    json.dump(validated_todo_dicts, f, indent=2)

print("Sample todo:")
print(json.dumps(sample_todo, indent=2))
print()
print("Validated", len(validated_todos), "todos")
print("Saved file:", Path("data/validated_todos.json").resolve())
```

</details>
